# 🐍 Day 3 — Python Internals, Visualization & Building Real Apps

**Course progression:** Day 1 (Fundamentals) → Day 2 (Pandas) → **Day 3 (You Are Here)**

Today we go from *knowing Python* to *understanding Python* and *shipping something real*.

| Block | Topic | Time |
|-------|-------|------|
| 1 | Tricky Concepts | 30 min |
| 2 | Decorators | 20 min |
| 3 | Standard Library | 15 min |
| 4 | Matplotlib | 15 min |
| 5 | Streamlit — Live Dashboard | 25 min |
| 🎁 Bonus | FastAPI + re / os / logging | 15 min |

---

# 🔵 Block 1 — Tricky Python Concepts (30 min)

These are the topics that trip up even experienced developers.
Understanding them separates **script writers** from **Python engineers**.

## 1.1 Mutable Default Arguments — The Classic Trap

**The bug:** Default argument values are evaluated **once** when the function is defined,
not every time the function is called. Mutable defaults (lists, dicts) persist across calls.

This is one of the most common Python interview questions.

In [ ]:
# ❌ BROKEN — list is created once, shared across all calls
def add_item_bad(item, cart=[]):
    cart.append(item)
    return cart

print(add_item_bad('apple'))   # ['apple']  ← looks fine
print(add_item_bad('banana'))  # ['apple', 'banana']  ← WHERE DID APPLE COME FROM?!
print(add_item_bad('cherry'))  # ['apple', 'banana', 'cherry']  ← completely wrong

print("---")

# ✅ FIXED — use None as sentinel, create fresh list inside the function
def add_item_good(item, cart=None):
    if cart is None:          # None is immutable, safe as default
        cart = []             # fresh list created on EACH call
    cart.append(item)
    return cart

print(add_item_good('apple'))   # ['apple']
print(add_item_good('banana'))  # ['banana']  ← independent, correct!
print(add_item_good('cherry'))  # ['cherry']

## 1.2 `is` vs `==` — Identity vs Equality

- `==` checks if two objects have the **same value**
- `is` checks if two variables point to the **exact same object in memory**

Python caches small integers (-5 to 256) and interned strings — this can fool you!

In [ ]:
# Lists — same value, different objects
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b)   # True  — same VALUES
print(a is b)   # False — different OBJECTS in memory
print(id(a), id(b))  # different memory addresses

print("---")

# Integer caching: Python caches -5 to 256
x = 100
y = 100
print(x is y)   # True  — same cached object (within -5..256)

x = 1000
y = 1000
print(x is y)   # False — outside cache range, new objects created

print("---")

# The ONLY correct use of 'is' in production code:
value = None
print(value is None)      # ✅ correct way to check for None
print(value == None)      # ⚠️  works but not Pythonic (PEP8 warning)

## 1.3 `*args` and `**kwargs` — Variable Arguments

- `*args` collects **positional** arguments into a **tuple**
- `**kwargs` collects **keyword** arguments into a **dict**

The names `args` and `kwargs` are convention — the `*` and `**` are what matter.

In [ ]:
# *args — accept any number of positional arguments
def total(*args):
    print(f"args is a {type(args).__name__}: {args}")
    return sum(args)

print(total(1, 2, 3))        # 6
print(total(10, 20, 30, 40)) # 100

print("---")

# **kwargs — accept any number of keyword arguments
def describe_person(**kwargs):
    print(f"kwargs is a {type(kwargs).__name__}: {kwargs}")
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

describe_person(name="Alice", age=30, city="Delhi")

print("---")

# Combined: regular + *args + **kwargs (order matters!)
def full_example(required, *args, **kwargs):
    print(f"required: {required}")
    print(f"extra positional: {args}")
    print(f"keyword options: {kwargs}")

full_example('hello', 1, 2, 3, color='red', size='large')

print("---")

# Unpacking with * and ** when CALLING a function
nums = [1, 2, 3]
print(total(*nums))          # unpacks list → total(1, 2, 3)

config = {'name': 'Bob', 'age': 25}
describe_person(**config)    # unpacks dict → describe_person(name='Bob', age=25)

## 1.4 List Comprehension vs Generator Expression

- **List comprehension** `[x for x in ...]` → builds the **entire list in memory at once**
- **Generator expression** `(x for x in ...)` → computes values **one at a time, on demand** (lazy)

Generators are memory-efficient for large datasets — critical for data engineering!

In [ ]:
import sys

# List comprehension — all values in memory at once
squares_list = [x**2 for x in range(1_000_000)]
print(f"List size:      {sys.getsizeof(squares_list):,} bytes")

# Generator expression — only one value in memory at a time
squares_gen = (x**2 for x in range(1_000_000))
print(f"Generator size: {sys.getsizeof(squares_gen):,} bytes")  # tiny!

print("---")

# Generators are iterators — consume once, then exhausted
gen = (x**2 for x in range(5))
print(list(gen))  # [0, 1, 4, 9, 16]
print(list(gen))  # [] ← exhausted! Can't reuse

print("---")

# When to use each:
# ✅ List comprehension: need to iterate multiple times, need indexing, small data
# ✅ Generator: large data, only iterating once, piping into sum/max/min/etc.

# Generator works perfectly with sum, any, all, max, min
total = sum(x**2 for x in range(1_000_000))  # no [] needed!
print(f"Sum of squares: {total:,}")

# Generator function with 'yield'
def fibonacci():
    a, b = 0, 1
    while True:          # infinite generator!
        yield a
        a, b = b, a + b

fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(f"First 10 Fibonacci: {first_10}")

## 1.5 Shallow Copy vs Deep Copy

Assignment (`=`) does **not** copy — it creates a new reference to the same object.

- **Shallow copy** — copies the outer container, but nested objects are still shared
- **Deep copy** — recursively copies everything; fully independent

In [ ]:
import copy

original = [[1, 2, 3], [4, 5, 6]]

# Assignment — just another name for the same object
alias = original
alias[0][0] = 99
print(f"original after alias change: {original}")  # [99, 2, 3] — CHANGED!

original = [[1, 2, 3], [4, 5, 6]]  # reset

# Shallow copy — new outer list, but inner lists are shared
shallow = copy.copy(original)
shallow[0][0] = 99      # modifies the shared inner list
print(f"original after shallow[0][0]=99: {original}")  # [99,2,3] — STILL affected!
shallow.append([7,8,9]) # adding to outer list is safe
print(f"original after shallow.append: {original}")    # unchanged ← outer is independent

original = [[1, 2, 3], [4, 5, 6]]  # reset

# Deep copy — fully independent
deep = copy.deepcopy(original)
deep[0][0] = 99
print(f"original after deep[0][0]=99: {original}")  # [1,2,3] — NOT affected ✅

# Rule of thumb:
# Flat structures → .copy() or [:] is fine
# Nested structures → always use copy.deepcopy()

## 1.6 `global` and `nonlocal`

Python uses **LEGB** scope: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

- `global` lets a function read/write a **module-level** variable
- `nonlocal` lets a nested function read/write a variable from its **enclosing function**

In [ ]:
# global
counter = 0

def increment():
    global counter          # without this, Python creates a NEW local 'counter'
    counter += 1

increment()
increment()
print(f"counter: {counter}")  # 2

print("---")

# nonlocal — for closures (nested functions)
def make_counter():
    count = 0               # lives in enclosing scope

    def increment():
        nonlocal count      # reach into enclosing function's scope
        count += 1
        return count

    return increment

my_counter = make_counter()
print(my_counter())  # 1
print(my_counter())  # 2
print(my_counter())  # 3
# Each call to make_counter() creates an independent counter — no global state!

## 1.7 Walrus Operator `:=` (Python 3.8+)

The walrus operator assigns **and** returns a value in a single expression.
Useful in `while` loops and comprehensions to avoid computing a value twice.

In [ ]:
import re

# Without walrus — compute twice, assign separately
data = "Error: file not found"
match = re.search(r'Error: (.+)', data)
if match:
    print(f"Found error: {match.group(1)}")

print("---")

# With walrus — assign and test in one step
if match := re.search(r'Error: (.+)', data):   # := assigns AND returns
    print(f"Found error: {match.group(1)}")

print("---")

# Classic use case: reading chunks in a while loop
import io
fake_file = io.StringIO("line1\nline2\nline3")

# Without walrus (repetitive)
line = fake_file.readline()
while line:
    print(f"  Read: {line.strip()}")
    line = fake_file.readline()

fake_file.seek(0)  # reset
print("---")

# With walrus (cleaner)
while line := fake_file.readline():
    print(f"  Read: {line.strip()}")

print("---")

# Walrus in list comprehension — avoid double computation
data = [1, -3, 5, -7, 9]
# Only compute abs() once per element, filter and use the computed value
results = [y for x in data if (y := abs(x)) > 3]
print(f"Absolute values > 3: {results}")  # [5, 7, 9]

---
# 🟡 Block 2 — Decorators (20 min)

A decorator is a function that **wraps another function** to extend or modify its behavior
without changing its source code. They're used everywhere: FastAPI routes, Django views,
pytest fixtures, caching, authentication, logging...

**Core idea:** functions are first-class objects in Python — they can be passed around and returned.

## 2.1 First-Class Functions — The Foundation

Before decorators, understand: functions can be stored in variables, passed as arguments,
and returned from other functions.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

# Store in variable
say_hello = greet
print(say_hello("Alice"))   # Hello, Alice!

# Pass as argument
def apply(func, value):
    return func(value)

print(apply(greet, "Bob"))  # Hello, Bob!

# Return from function
def make_multiplier(n):
    def multiply(x):        # inner function captures 'n' — this is a closure
        return x * n
    return multiply         # return the function itself, not the result

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5))  # 10
print(triple(5))  # 15

## 2.2 Basic Decorator — How It Works Under the Hood

A decorator is just a function that takes a function and returns a (usually modified) function.
The `@decorator` syntax is **syntactic sugar** for `func = decorator(func)`.

In [ ]:
# Manual decorator (no @ syntax) — shows exactly what's happening
def shout(func):                 # decorator: takes a function
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()    # modify the result
    return wrapper               # return the new function

def greet(name):
    return f"hello, {name}"

greet = shout(greet)            # manual decoration — greet is now the wrapper
print(greet("alice"))           # HELLO, ALICE

print("---")

# Same thing with @ syntax — cleaner, idiomatic
@shout
def farewell(name):
    return f"goodbye, {name}"

# @shout is EXACTLY equivalent to: farewell = shout(farewell)
print(farewell("bob"))          # GOODBYE, BOB

print("---")

# Practical decorator: timing a function
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"[timer] {func.__name__} took {end - start:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(10_000_000))

## 2.3 `functools.wraps` — Preserving Metadata

When you wrap a function, the wrapper hides the original function's `__name__`, `__doc__`, etc.
`@functools.wraps(func)` copies that metadata to the wrapper — **always use it in production**.

In [ ]:
from functools import wraps

# Without @wraps — metadata is lost
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    """This does something important."""
    pass

print(f"name: {my_func.__name__}")  # 'wrapper' — WRONG
print(f"doc:  {my_func.__doc__}")   # None — WRONG (breaks help())

print("---")

# With @wraps — metadata preserved
def good_decorator(func):
    @wraps(func)                    # copies __name__, __doc__, __annotations__...
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def my_func():
    """This does something important."""
    pass

print(f"name: {my_func.__name__}")  # 'my_func' ✅
print(f"doc:  {my_func.__doc__}")   # 'This does something important.' ✅

## 2.4 Decorator with Arguments — Factory Pattern

When you want to configure your decorator (e.g., `@retry(times=3)`), you need an extra
outer layer: a function that **returns** a decorator.

In [ ]:
from functools import wraps

# Decorator factory — takes config arguments, returns a decorator
def repeat(times=2):
    """Run a function 'times' times."""
    def decorator(func):           # this is the actual decorator
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                result = func(*args, **kwargs)
            return result          # return last result
        return wrapper
    return decorator               # return the decorator

@repeat(times=3)
def say(message):
    print(f"  >> {message}")

say("hello")  # prints 3 times

print("---")

# Real-world example: retry on exception
import time

def retry(times=3, delay=0.5, exceptions=(Exception,)):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt < times:
                        time.sleep(delay)
            raise RuntimeError(f"{func.__name__} failed after {times} attempts")
        return wrapper
    return decorator

attempt_count = 0

@retry(times=3, delay=0.1)
def flaky_api_call():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError("Server busy")
    return "Success!"

print(flaky_api_call())

## 2.5 Stacking Decorators

Multiple decorators can be stacked. They apply **bottom-up** (innermost first),
but the execution order feels **top-down**.

In [ ]:
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold       # applied second (outer)
@italic     # applied first (inner)
def say(text):
    return text

# Equivalent to: say = bold(italic(say))
# Execution: bold wraps italic wraps say
print(say("hello"))  # <b><i>hello</i></b>

print("---")

# Combine timer + logger decorators
import time

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__} with args={args} kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result}")
        return result
    return wrapper

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[TIMER] {func.__name__}: {time.perf_counter()-start:.4f}s")
        return result
    return wrapper

@log_call
@timer
def compute(n):
    return sum(range(n))

compute(1_000_000)

## 2.6 `@property` — Managed Attributes

The `@property` decorator turns a method into a computed attribute.
Use it to add validation, computed values, or read-only attributes to classes.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius      # _ prefix = private by convention

    @property
    def celsius(self):
        """Getter — accessed as attribute, not method."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """Setter with validation."""
        if value < -273.15:
            raise ValueError(f"Temperature {value}°C is below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self):
        """Computed property — read only (no setter)."""
        return self._celsius * 9/5 + 32

    @property
    def kelvin(self):
        return self._celsius + 273.15

t = Temperature(100)
print(f"Celsius:    {t.celsius}°C")      # accessed like attribute
print(f"Fahrenheit: {t.fahrenheit}°F")   # computed on the fly
print(f"Kelvin:     {t.kelvin}K")

t.celsius = 0                             # setter called with validation
print(f"After set:  {t.celsius}°C = {t.fahrenheit}°F")

try:
    t.celsius = -999                      # triggers validation error
except ValueError as e:
    print(f"Error: {e}")

---
# 🟢 Block 3 — Standard Library Essentials (15 min)

Python's stdlib is massive. These 5 modules are the ones you'll use daily as a data engineer or backend developer.

## 3.1 `datetime` — Dates and Times

Date handling is unavoidable in data work. `datetime` covers parsing, formatting, arithmetic, and timezones.

In [ ]:
from datetime import datetime, date, timedelta

# Current date and time
now = datetime.now()
today = date.today()
print(f"now:   {now}")
print(f"today: {today}")

# Formatting — strftime (string from time)
print(now.strftime("%Y-%m-%d"))           # '2024-01-15'
print(now.strftime("%d/%m/%Y %H:%M"))     # '15/01/2024 14:30'
print(now.strftime("%B %d, %Y"))          # 'January 15, 2024'

# Parsing — strptime (string parse time)
date_str = "2024-03-15"
parsed = datetime.strptime(date_str, "%Y-%m-%d")
print(f"Parsed: {parsed}  Type: {type(parsed).__name__}")

# Date arithmetic with timedelta
tomorrow = today + timedelta(days=1)
last_week = today - timedelta(weeks=1)
print(f"Tomorrow:  {tomorrow}")
print(f"Last week: {last_week}")

# Calculate difference between dates
start = date(2024, 1, 1)
end = date(2024, 12, 31)
diff = end - start
print(f"Days in 2024: {diff.days}")

## 3.2 `collections` — Specialized Data Structures

The `collections` module has high-performance alternatives to built-in types.

In [ ]:
from collections import Counter, defaultdict, namedtuple, deque

# Counter — frequency counting (great for text/categorical data)
words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']
counts = Counter(words)
print(f"Counts:       {counts}")
print(f"Most common:  {counts.most_common(2)}")  # top 2
print(f"apple count:  {counts['apple']}")
print(f"missing key:  {counts['mango']}")        # 0, not KeyError!

print("---")

# defaultdict — dict with a default value factory
# Without defaultdict: if key missing → KeyError
# With defaultdict:    if key missing → call the factory

# Group items by first letter
words = ['apple', 'avocado', 'banana', 'blueberry', 'cherry']
groups = defaultdict(list)    # missing key → empty list
for word in words:
    groups[word[0]].append(word)
print(f"Groups: {dict(groups)}")

print("---")

# namedtuple — lightweight, immutable record
Point = namedtuple('Point', ['x', 'y'])
p = Point(3, 4)
print(f"Point: {p}")
print(f"x={p.x}, y={p.y}")
dist = (p.x**2 + p.y**2)**0.5
print(f"Distance from origin: {dist}")

print("---")

# deque — double-ended queue, O(1) for append/pop from both ends
# vs list: list.pop(0) is O(n), deque.popleft() is O(1)
dq = deque([1, 2, 3], maxlen=5)  # maxlen auto-removes oldest when full
dq.appendleft(0)
dq.append(4)
dq.append(5)   # pushes out 0 because maxlen=5
print(f"deque: {dq}")

## 3.3 `itertools` — Lazy Iterator Combinators

`itertools` gives you composable, memory-efficient tools for working with sequences.
These are bread-and-butter for data pipelines.

In [ ]:
import itertools

# chain — flatten multiple iterables into one
result = list(itertools.chain([1, 2], [3, 4], [5, 6]))
print(f"chain:  {result}")

# chain.from_iterable — flatten a list of lists
nested = [[1, 2], [3, 4], [5, 6]]
flat = list(itertools.chain.from_iterable(nested))
print(f"flatten: {flat}")

# islice — lazy slicing without materializing
first_5 = list(itertools.islice(range(1_000_000), 5))
print(f"islice: {first_5}")

# groupby — group consecutive equal elements (sort first!)
data = sorted([('a', 1), ('b', 2), ('a', 3), ('b', 4)], key=lambda x: x[0])
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f"  {key}: {list(group)}")

# product — Cartesian product (nested loop replacement)
colors = ['red', 'blue']
sizes = ['S', 'M', 'L']
skus = list(itertools.product(colors, sizes))
print(f"SKUs: {skus}")

# combinations and permutations
items = ['A', 'B', 'C']
print(f"combinations(2): {list(itertools.combinations(items, 2))}")
print(f"permutations(2): {list(itertools.permutations(items, 2))}")

## 3.4 `pathlib` — Modern File System Operations

`pathlib.Path` is the modern replacement for `os.path`. It uses `/` as the path separator
on all operating systems, and everything is an object with methods.

In [ ]:
from pathlib import Path

# Create path objects (doesn't create files/dirs)
home = Path.home()
cwd = Path.cwd()
print(f"Home: {home}")
print(f"CWD:  {cwd}")

# Build paths with / operator — works on Windows AND Mac/Linux!
data_dir = cwd / 'data' / 'processed'
output = data_dir / 'report.csv'
print(f"Output path: {output}")

# Path properties
p = Path('/home/alice/data/sales.csv')
print(f"name:    {p.name}")
print(f"stem:    {p.stem}")
print(f"suffix:  {p.suffix}")
print(f"parent:  {p.parent}")
print(f"parts:   {p.parts}")

# File operations
p_tmp = Path('/tmp/test_pathlib.txt')
p_tmp.write_text("Hello from pathlib!")  # write string
content = p_tmp.read_text()              # read string
print(f"Content: {content}")
p_tmp.unlink()                           # delete file

# Glob — find files by pattern
py_files = list(cwd.glob('**/*.py'))     # recursive glob
print(f"Python files in cwd: {len(py_files)}")

# Create directories safely
new_dir = Path('/tmp/day3_test/subdir')
new_dir.mkdir(parents=True, exist_ok=True)  # -p flag equivalent
print(f"Created: {new_dir.exists()}")
new_dir.rmdir()                              # remove
new_dir.parent.rmdir()

## 3.5 `json` — Data Serialization

JSON is the universal language of APIs and config files.
`json` module handles encoding (Python → JSON string) and decoding (JSON string → Python).

In [ ]:
import json
from pathlib import Path

# Python dict → JSON string
data = {
    'name': 'Alice',
    'age': 30,
    'scores': [95, 87, 92],
    'active': True,
    'address': None
}

json_str = json.dumps(data, indent=2)   # indent for pretty print
print(json_str)

print("---")

# JSON string → Python dict
parsed = json.loads(json_str)
print(f"Type: {type(parsed).__name__}")
print(f"Name: {parsed['name']}, Age: {parsed['age']}")

print("---")

# Write to / read from file
p = Path('/tmp/data.json')
with open(p, 'w') as f:
    json.dump(data, f, indent=2)    # dump to file

with open(p) as f:
    loaded = json.load(f)           # load from file

print(f"Loaded: {loaded}")
p.unlink()

print("---")

# Custom serialization for types JSON doesn't support
from datetime import date

class DateEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, date):
            return obj.isoformat()   # date → 'YYYY-MM-DD' string
        return super().default(obj)

event = {'name': 'Launch', 'date': date(2024, 3, 15)}
print(json.dumps(event, cls=DateEncoder))  # handles date object

---
# 🟠 Block 4 — Matplotlib Essentials (15 min)

We'll cover the **5 most useful chart types** plus formatting and saving.
Goal: produce clean, publication-ready charts fast — not memorize every parameter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Consistent style across all plots today
plt.style.use('seaborn-v0_8-whitegrid')

# ── Sales data we'll use throughout ──
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sales   = [12, 19, 15, 22, 28, 35, 31, 40, 38, 45, 50, 62]
returns = [ 2,  3,  1,  4,  5,  6,  4,  7,  5,  8,  9, 10]

print("Data loaded. Let's plot!")

## 4.1 Line Chart — `plt.plot()`
Best for: trends over time, continuous data.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))   # always use figure + axes objects

ax.plot(months, sales,   marker='o', color='steelblue', linewidth=2, label='Sales')
ax.plot(months, returns, marker='s', color='tomato',    linewidth=2, label='Returns', linestyle='--')

# Formatting
ax.set_title('Monthly Sales vs Returns', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Units')
ax.legend()
ax.set_ylim(0)          # start y-axis at 0

plt.tight_layout()       # prevent label clipping
plt.show()

## 4.2 Bar Chart — `plt.bar()`
Best for: comparing categories, ranked lists.

In [ ]:
categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports']
revenue    = [45_000, 32_000, 28_000, 15_000, 22_000]
colors_map = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(categories, revenue, color=colors_map, edgecolor='white', linewidth=0.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 500,
            f'${height:,.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Revenue by Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue ($)')
ax.set_ylim(0, max(revenue) * 1.15)   # room for labels

plt.tight_layout()
plt.show()

## 4.3 Scatter Plot — `plt.scatter()`
Best for: relationships between two numeric variables, outlier detection.

In [ ]:
np.random.seed(42)
n = 100
advertising = np.random.uniform(10, 100, n)
noise = np.random.normal(0, 10, n)
sales_scatter = advertising * 1.8 + noise + 20   # sales ≈ 1.8 × advertising

fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(advertising, sales_scatter,
                c=sales_scatter,       # color by value
                cmap='viridis',
                alpha=0.7,
                s=60,                  # marker size
                edgecolors='white', linewidth=0.5)

# Add trend line
z = np.polyfit(advertising, sales_scatter, 1)
p = np.poly1d(z)
x_line = np.linspace(advertising.min(), advertising.max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.8, label=f'Trend (slope={z[0]:.1f})')

plt.colorbar(sc, label='Sales')
ax.set_title('Advertising Spend vs Sales', fontsize=14, fontweight='bold')
ax.set_xlabel('Advertising Spend ($)')
ax.set_ylabel('Sales ($)')
ax.legend()

plt.tight_layout()
plt.show()

## 4.4 Histogram — `plt.hist()`
Best for: understanding the distribution of a single numeric variable.

In [ ]:
np.random.seed(0)
order_values = np.concatenate([
    np.random.normal(50, 15, 500),   # typical orders
    np.random.normal(200, 30, 100),  # high-value orders
])
order_values = order_values[order_values > 0]   # no negatives

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(order_values, bins=40, color='steelblue', edgecolor='white',
        alpha=0.8, density=False)

# Annotate mean and median
mean_val   = order_values.mean()
median_val = np.median(order_values)
ax.axvline(mean_val,   color='red',    linestyle='--', linewidth=1.5, label=f'Mean ${mean_val:.1f}')
ax.axvline(median_val, color='orange', linestyle='-',  linewidth=1.5, label=f'Median ${median_val:.1f}')

ax.set_title('Distribution of Order Values', fontsize=14, fontweight='bold')
ax.set_xlabel('Order Value ($)')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.show()

## 4.5 Subplots + Saving — `plt.subplot()` & `savefig()`

Subplots let you show multiple charts in one figure.
`savefig()` exports to PNG, PDF, SVG — production-ready.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # 2 rows, 2 cols
fig.suptitle('Sales Dashboard', fontsize=16, fontweight='bold', y=1.02)

# Plot 1: Line
axes[0, 0].plot(months, sales, marker='o', color='steelblue')
axes[0, 0].set_title('Monthly Sales')
axes[0, 0].set_ylabel('Units')

# Plot 2: Bar
axes[0, 1].bar(months, returns, color='tomato', alpha=0.8)
axes[0, 1].set_title('Monthly Returns')

# Plot 3: Scatter
axes[1, 0].scatter(returns, sales, c='purple', alpha=0.7)
axes[1, 0].set_title('Returns vs Sales')
axes[1, 0].set_xlabel('Returns')
axes[1, 0].set_ylabel('Sales')

# Plot 4: Histogram
axes[1, 1].hist(sales, bins=8, color='green', alpha=0.7, edgecolor='white')
axes[1, 1].set_title('Sales Distribution')
axes[1, 1].set_xlabel('Units')

plt.tight_layout()

# Save — dpi=150 is good for presentations, dpi=300 for print
output_path = '/tmp/sales_dashboard.png'
fig.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"Saved to {output_path}")
plt.show()

---
# 🔴 Block 5 — Streamlit: Build a Live Data Dashboard (25 min)

**Why Streamlit?** It turns a Python script into a shareable web app with almost no extra code.
Your Pandas + Matplotlib skills from Day 2 & Block 4 are directly reusable here.

> ⚠️ **Streamlit cannot run inside Jupyter.** The cells below show the code.
> Save the final cell to `dashboard.py` and run with `streamlit run dashboard.py`

## 5.1 Core Streamlit Primitives

The three most-used display functions — enough to build 80% of dashboards.

In [ ]:
# ── Run this as a .py file, not in Jupyter ──
# streamlit run 5_1_basics.py

STREAMLIT_BASICS = '''
import streamlit as st
import pandas as pd

# ── Layout ──
st.title("My First Streamlit App")        # big heading
st.header("Section Header")               # h2
st.subheader("Subsection")                # h3
st.write("Hello! This auto-detects type.")  # renders text, df, dict, chart...
st.markdown("**Bold** and *italic* text")   # full markdown support

# ── Data display ──
df = pd.DataFrame({'A': [1,2,3], 'B': [4,5,6]})
st.dataframe(df)           # interactive scrollable table
st.table(df)               # static table
st.json({'key': 'value'})  # formatted JSON viewer

# ── Metrics (KPI cards) ──
col1, col2, col3 = st.columns(3)
col1.metric("Revenue",  "$45,000", "+12%")
col2.metric("Orders",   "1,234",   "-3%")
col3.metric("NPS Score","72",      "+5pts")
'''

print(STREAMLIT_BASICS)

## 5.2 Built-in Charts + Embedding Matplotlib

Streamlit has 1-line chart functions AND can embed your Matplotlib figures.

In [ ]:
STREAMLIT_CHARTS = '''
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.DataFrame({
    'Month': ['Jan','Feb','Mar','Apr','May','Jun'],
    'Sales': [12, 19, 15, 22, 28, 35],
    'Returns': [2, 3, 1, 4, 5, 6]
}).set_index('Month')

st.subheader("Built-in Charts (1-liner each)")
st.line_chart(df[['Sales', 'Returns']])    # instant interactive line chart
st.bar_chart(df['Sales'])                  # instant bar chart
st.area_chart(df)                          # area chart

st.subheader("Embedded Matplotlib")
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df['Returns'], df['Sales'], c='steelblue', s=80)
ax.set_xlabel('Returns')
ax.set_ylabel('Sales')
ax.set_title('Returns vs Sales')
st.pyplot(fig)                             # embed any matplotlib figure
plt.close(fig)                             # always close after st.pyplot()
'''

print(STREAMLIT_CHARTS)

## 5.3 Widgets — Making It Interactive

Every widget returns its current value. Streamlit reruns the entire script on each interaction.

In [ ]:
STREAMLIT_WIDGETS = '''
import streamlit as st

# Input widgets — all return their current value
name     = st.text_input("Your name", value="Alice")
age      = st.slider("Age", min_value=0, max_value=100, value=25)
category = st.selectbox("Category", ["Electronics", "Clothing", "Food"])
options  = st.multiselect("Pick multiple", ["A", "B", "C", "D"])
checked  = st.checkbox("Include returns?")
date_val = st.date_input("Select date")

# Sidebar widgets
with st.sidebar:
    st.header("Filters")
    min_val = st.slider("Min sales", 0, 100, 10)
    max_val = st.slider("Max sales", 0, 100, 50)

# Use widget values to filter data
st.write(f"Hello {name}! You selected {category} with min={min_val}, max={max_val}")

# File uploader
uploaded = st.file_uploader("Upload a CSV", type=["csv"])
if uploaded:
    import pandas as pd
    df = pd.read_csv(uploaded)
    st.dataframe(df.head())
    st.success(f"Loaded {len(df)} rows!")
'''

print(STREAMLIT_WIDGETS)

## 5.4 🎯 Live Demo — Full Sales Dashboard

This is the complete dashboard to build live in class.
Save this to `dashboard.py` and run `streamlit run dashboard.py`

In [ ]:
DASHBOARD_APP = '''
# dashboard.py — run with: streamlit run dashboard.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Page config ──
st.set_page_config(page_title="Sales Dashboard", layout="wide", page_icon="📊")

# ── Data (reuse Day 2 sales DataFrame) ──
@st.cache_data                     # cache so we don't reload on every interaction
def load_data():
    data = {
        'Month':    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
        'Sales':    [12, 19, 15, 22, 28, 35, 31, 40, 38, 45, 50, 62],
        'Returns':  [2,  3,  1,  4,  5,  6,  4,  7,  5,  8,  9, 10],
        'Category': ['A','B','A','C','B','A','C','B','A','C','B','A']
    }
    df = pd.DataFrame(data)
    df['Net'] = df['Sales'] - df['Returns']
    df['Return Rate (%)'] = (df['Returns'] / df['Sales'] * 100).round(1)
    return df

df = load_data()

# ── Sidebar filters ──
with st.sidebar:
    st.header("⚙️ Filters")
    selected_cats = st.multiselect("Category", df['Category'].unique(),
                                   default=df['Category'].unique())
    month_range = st.slider("Month range", 1, 12, (1, 12))

# Apply filters
mask = (df['Category'].isin(selected_cats) &
        df.index.map(lambda i: month_range[0]-1 <= i <= month_range[1]-1))
filtered = df[mask]

# ── Header ──
st.title("📊 Sales Dashboard")
st.caption(f"Showing {len(filtered)} of {len(df)} months")

# ── KPI cards ──
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Sales",   f"{filtered['Sales'].sum():,}")
col2.metric("Total Returns", f"{filtered['Returns'].sum():,}")
col3.metric("Net Sales",     f"{filtered['Net'].sum():,}")
col4.metric("Avg Return Rate", f"{filtered['Return Rate (%)'].mean():.1f}%")

st.divider()

# ── Charts ──
left, right = st.columns(2)

with left:
    st.subheader("Sales Trend")
    st.line_chart(filtered.set_index('Month')[['Sales', 'Returns', 'Net']])

with right:
    st.subheader("Returns by Month")
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(filtered['Month'], filtered['Return Rate (%)'], color='tomato', alpha=0.8)
    ax.set_ylabel('Return Rate (%)')
    ax.set_ylim(0, 25)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close(fig)

# ── Raw data ──
with st.expander("📋 Raw Data"):
    st.dataframe(filtered, use_container_width=True)
    csv = filtered.to_csv(index=False)
    st.download_button("Download CSV", csv, "sales.csv", "text/csv")
'''

# Save to file so students can run it
with open('/tmp/dashboard.py', 'w') as f:
    f.write(DASHBOARD_APP)

print("✅ Saved to /tmp/dashboard.py")
print("Run with: streamlit run /tmp/dashboard.py")

---
# 🎁 Bonus Block — FastAPI + `re` / `os` / `logging`

## Why FastAPI over argparse?
- `argparse` is CLI-only and rarely used outside scripting
- FastAPI is in-demand for ML APIs, data services, and microservices
- It showcases your **Block 2 decorators** (`@app.get`) in a production context
- You get interactive auto-docs at `/docs` for free

## B.1 FastAPI — Your First API in 5 Lines

In [ ]:
# Install: pip install fastapi uvicorn
# Run:     uvicorn api:app --reload

FASTAPI_BASIC = '''
# api.py
from fastapi import FastAPI

app = FastAPI(title="Day 3 API", description="Sales data API", version="1.0")

# ── Routes ──
@app.get("/")                          # GET /
def root():
    return {"message": "API is running"}

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.get("/items/{item_id}")           # path parameter
def get_item(item_id: int, q: str = None):  # query param: /items/5?q=hello
    return {"item_id": item_id, "query": q}

@app.post("/items")                    # POST with request body
def create_item(item: dict):
    # In production: use Pydantic models instead of plain dict
    return {"created": item, "status": "ok"}
'''
print(FASTAPI_BASIC)

## B.2 FastAPI + Pydantic + Pandas — A Real Data API

In [ ]:
FASTAPI_FULL = '''
# sales_api.py — run with: uvicorn sales_api:app --reload
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import pandas as pd

app = FastAPI(title="Sales API")

# ── Data (in production this would be a database) ──
df = pd.DataFrame({
    'id':       [1, 2, 3, 4, 5],
    'product':  ['Laptop', 'Phone', 'Tablet', 'Watch', 'Earbuds'],
    'sales':    [450, 820, 310, 240, 670],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Wearables', 'Audio']
})

# ── Pydantic model for request validation ──
class SaleRecord(BaseModel):
    product:  str
    sales:    int
    category: str

# ── Endpoints ──
@app.get("/sales")                              # GET /sales?category=Electronics
def get_sales(category: Optional[str] = None):
    result = df.copy()
    if category:
        result = result[result['category'] == category]
    if result.empty:
        raise HTTPException(status_code=404, detail=f"No sales for category: {category}")
    return result.to_dict(orient="records")

@app.get("/sales/summary")
def sales_summary():
    return {
        "total_sales":    int(df['sales'].sum()),
        "avg_sales":      float(df['sales'].mean()),
        "top_product":    df.loc[df['sales'].idxmax(), 'product'],
        "by_category":    df.groupby('category')['sales'].sum().to_dict()
    }

@app.get("/sales/{product_id}")
def get_product(product_id: int):
    row = df[df['id'] == product_id]
    if row.empty:
        raise HTTPException(status_code=404, detail="Product not found")
    return row.iloc[0].to_dict()

@app.post("/sales", status_code=201)
def add_sale(record: SaleRecord):
    global df
    new_id = df['id'].max() + 1
    new_row = pd.DataFrame([{'id': new_id, **record.dict()}])
    df = pd.concat([df, new_row], ignore_index=True)
    return {"id": new_id, **record.dict()}

# Visit http://localhost:8000/docs for interactive Swagger UI!
'''
print(FASTAPI_FULL)

## B.3 `re` — Regular Expressions

Essential for text parsing, data cleaning, log parsing.

In [ ]:
import re

# Core functions:
# re.search()  — find first match anywhere in string → Match or None
# re.match()   — match only at START of string
# re.findall() — return list of ALL matches
# re.sub()     — replace matches with new string
# re.split()   — split string by pattern

text = "Contact us at support@company.com or sales@company.org by 2024-03-15"

# Find all emails
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', text)
print(f"Emails:  {emails}")

# Find dates (YYYY-MM-DD)
dates = re.findall(r'\d{4}-\d{2}-\d{2}', text)
print(f"Dates:   {dates}")

# Extract groups with search
m = re.search(r'(\d{4})-(\d{2})-(\d{2})', text)
if m:
    print(f"Year: {m.group(1)}, Month: {m.group(2)}, Day: {m.group(3)}")

# Clean data — remove special characters
dirty = "  Hello,   World!!! 123  "
clean = re.sub(r'[^a-zA-Z0-9 ]', '', dirty).strip()
clean = re.sub(r'\s+', ' ', clean)          # collapse multiple spaces
print(f"Cleaned: '{clean}'")

# Parse log lines
log = '2024-01-15 14:32:01 ERROR api.py:42 Connection timeout after 30s'
pattern = r'(?P<date>[\d-]+) (?P<time>[\d:]+) (?P<level>\w+) (?P<location>\S+) (?P<message>.+)'
m = re.match(pattern, log)
if m:
    print(f"Level: {m.group('level')}, Message: {m.group('message')}")

## B.4 `os` — Operating System Interface

In [ ]:
import os

# Environment variables — critical for secrets management
os.environ['MY_API_KEY'] = 'secret123'  # set (usually you'd read, not set)
api_key = os.environ.get('MY_API_KEY', 'default')  # .get() avoids KeyError
print(f"API Key: {api_key}")

# Path operations (prefer pathlib, but os.path is everywhere)
path = '/home/alice/data/sales.csv'
print(f"basename: {os.path.basename(path)}")
print(f"dirname:  {os.path.dirname(path)}")
print(f"exists:   {os.path.exists(path)}")
print(f"join:     {os.path.join('/home/alice', 'data', 'sales.csv')}")

# Directory operations
print(f"cwd: {os.getcwd()}")
os.makedirs('/tmp/day3_test/subdir', exist_ok=True)
print(f"Created dir, exists: {os.path.exists('/tmp/day3_test/subdir')}")

# Walk directory tree — find all files
for root, dirs, files in os.walk('/tmp'):
    level = root.replace('/tmp', '').count(os.sep)
    if level < 2:   # limit depth
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        break

## B.5 `logging` — Production-Grade Logging

> **Never use `print()` in production code.** Use `logging`.

Logging gives you: severity levels, timestamps, file output, log filtering, and structured formatting — all without changing your code logic.

In [ ]:
import logging

# ── Basic configuration ──
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logger = logging.getLogger('day3')   # name your logger (use module name in prod)

# Five severity levels (ascending)
logger.debug('Variable x=42 — only shown in development')  # DEBUG
logger.info('Pipeline started with 1000 rows')              # INFO
logger.warning('Missing values detected: 5 rows')           # WARNING
logger.error('Failed to connect to database')               # ERROR
logger.critical('Data corruption detected — halting')        # CRITICAL

print("---")

# ── File + console logging (production pattern) ──
def get_logger(name, log_file='/tmp/app.log'):
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s'
    )

    # Console handler — INFO and above
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(formatter)

    # File handler — DEBUG and above (all messages)
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(formatter)

    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

prod_logger = get_logger('sales_pipeline')
prod_logger.info('Processing started')
prod_logger.debug('Internal state: queue_size=42')  # only in file
prod_logger.warning('Slow query detected: 2.3s')

print(f"\nLog file at /tmp/app.log — debug messages only visible there")

---
# 📋 Day 3 Summary

| Block | What You Learned | Key Takeaway |
|-------|------------------|--------------|
| 1 — Tricky Concepts | Mutable defaults, `is` vs `==`, `*args/**kwargs`, generators, shallow/deep copy, `global`/`nonlocal`, walrus | Know *why* Python behaves the way it does |
| 2 — Decorators | Basic deco, `@wraps`, deco with args, stacking, `@property` | Decorators = functions that modify functions |
| 3 — stdlib | `datetime`, `collections`, `itertools`, `pathlib`, `json` | Stop reinventing the wheel |
| 4 — Matplotlib | `plot`, `bar`, `scatter`, `hist`, `subplot` + `savefig` | 5 chart types cover 90% of cases |
| 5 — Streamlit | `st.write/dataframe/chart/pyplot`, widgets, full dashboard | Pandas + Matplotlib → shareable web app |
| Bonus — FastAPI | `@app.get/post`, Pydantic, data API + `re`/`os`/`logging` | Production skills that employers want |

**Next: Day 4 — EDA & Statistical Analysis** (Seaborn + advanced Pandas groupby/pivot)

---
*Day 3 | Python Internals, Visualization & Real Apps*